## Support Vector Machine (SVM)

**Основные цели:**
- Написать свою реализацию SVM
- Решить задачу используя готовые реализации SVM
- Сравнить результат с Нейронными Сетями

In [ ]:
# Скачиваем dataset
# Задача - Binary Classification
# Dataset - Smoke Detection Dataset
# Количество данных - 62630

import kaggle

kaggle.api.dataset_download_files(
    "deepcontractor/smoke-detection-dataset",
    path="./data",
    unzip=True
)

Dataset URL: https://www.kaggle.com/datasets/deepcontractor/smoke-detection-dataset


In [1]:
# Импортируем нужные библиотеки
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score

In [2]:
df = pd.read_csv(".\data\smoke_detection_iot.csv", index_col=0)
df = df.drop(columns=['UTC', 'CNT'])
print(f"shape of the df: {df.shape}")

shape of the df: (62630, 13)


In [3]:
df.head()

,Temperature[C],Humidity[%],TVOC[ppb],eCO2[ppm],Raw H2,Raw Ethanol,Pressure[hPa],PM1.0,PM2.5,NC0.5,NC1.0,NC2.5,Fire Alarm
0,20.000,57.36,0,400,12306,18520,939.735,0.0,0.0,0.0,0.0,0.0,0
1,20.015,56.67,0,400,12345,18651,939.744,0.0,0.0,0.0,0.0,0.0,0
2,20.029,55.96,0,400,12374,18764,939.738,0.0,0.0,0.0,0.0,0.0,0
3,20.044,55.28,0,400,12390,18849,939.736,0.0,0.0,0.0,0.0,0.0,0
4,20.059,54.69,0,400,12403,18921,939.744,0.0,0.0,0.0,0.0,0.0,0


In [ ]:
df['Fire Alarm'] = df['Fire Alarm'].apply(lambda x: -1 if x == 0 else 1) # Как написано в лекции y -> (-1, 1)
df.head()

,Temperature[C],Humidity[%],TVOC[ppb],eCO2[ppm],Raw H2,Raw Ethanol,Pressure[hPa],PM1.0,PM2.5,NC0.5,NC1.0,NC2.5,Fire Alarm
0,20.000,57.36,0,400,12306,18520,939.735,0.0,0.0,0.0,0.0,0.0,-1
1,20.015,56.67,0,400,12345,18651,939.744,0.0,0.0,0.0,0.0,0.0,-1
2,20.029,55.96,0,400,12374,18764,939.738,0.0,0.0,0.0,0.0,0.0,-1
3,20.044,55.28,0,400,12390,18849,939.736,0.0,0.0,0.0,0.0,0.0,-1
4,20.059,54.69,0,400,12403,18921,939.744,0.0,0.0,0.0,0.0,0.0,-1


### Готовая реализация -> Параметры модели -> Реализация

#### **1) Готовая реализация**

In [9]:
y = df['Fire Alarm'].copy()
X = df.drop(columns=['Fire Alarm']).copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (41962, 12)
X_test shape: (20668, 12)


In [10]:
X_train = X_train.to_numpy()
y_train = y_train.to_numpy()
X_test = X_test.to_numpy()
y_test = y_test.to_numpy()

Наш **pipeline** будет содержать следующие этапы:
* **StandardScaler()** -  Котегорически нельзя забывать про Стандартизацию данных
* **SVC()** - Реализация SVM в sklearn | **kernel = 'rbf'** (Гаусовский) | **С** - параметр регулиризации | **gamma** - параметр для высчитывания ядер

In [34]:
# Наш pipeline
my_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('SVM', SVC(kernel='rbf', C=2.0, gamma=0.02))
]) 

In [35]:
my_pipeline.fit(X_train, y_train) # Обучаем модель

,steps,"[('scaler', ...), ('SVM', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,C,2.0
,kernel,'rbf'
,degree,3
,gamma,0.02


In [36]:
y_test_pred = my_pipeline.predict(X_test)
print(f"Accuracy score: {round(accuracy_score(y_test, y_test_pred), 4)}")
print(f"F1 score: {round(f1_score(y_test, y_test_pred), 4)}")
print(f"Confusion matrix: \n{confusion_matrix(y_test, y_test_pred)}")

Accuracy score: 0.953
F1 score: 0.9679
Confusion matrix: 
[[ 5026   872]
 [  100 14670]]


#### **2) Параметры модели**

**Формула для predict**:

$$
f(x) = \sum_{i=1}^{n} \alpha_i y_i K(x_i, x) + b
$$

**RBF kernel**:
$$
K(x_i, x) = \exp(-\gamma \|x_i - x\|^2)
$$

**Класс предсказания**:
$$
\hat{y} = \text{sign}(f(x))
$$

In [37]:
svc = my_pipeline.named_steps["SVM"]

x_i = svc.support_vectors_ # x_i - тренировочные данные
dual_coef = svc.dual_coef_ # α_i * y_i
dual_coef = dual_coef[0]
b = svc.intercept_ # b
gamma = svc.gamma # gamma

#### **3) Реализация**

In [38]:
def kernel(x, z, gamma):
    x = np.array(x)
    z = np.array(z)
    return np.exp(-gamma * np.linalg.norm(x - z)**2)

def predict(x, X_sv, dual_coef, b, gamma):
    s = 0
    for i in range(len(X_sv)):
        s += dual_coef[i] * kernel(x, X_sv[i], gamma)
    
    return s + b

In [53]:
index = 3
scaler = my_pipeline["scaler"]
x = scaler.transform([X_test[index]])[0]
X_sv = x_i

pred = predict(x, X_sv, dual_coef, b, gamma)

print(f"predict: {pred}")

if pred > 0:
    print(f"class: {1}")
elif pred < 0:
    print(f"class: {-1}")
else:
    print("Class into margin!!!")
    
print(f"y_true = {y_test[index]}")

predict: [3.05824202]
class: 1
y_true = 1


При обучении SVM в **dual-форме** оптимизируется двойственная **QP (Quadratic Programming)** задача по переменным $\alpha_i$ с ограничениями:

$$
0 \le \alpha_i \le C
$$

и

$$
\sum_{i=1}^{n} \alpha_i y_i = 0
$$

Алгоритм **SMO (Sequential Minimal Optimization)** решает эту задачу итеративно, обновляя по две переменные $(\alpha_i, \alpha_j)$ за раз.  
Это позволяет сохранять ограничение:

$$
\sum \alpha_i y_i = 0
$$

и получить **аналитическое локальное обновление**.

Этапы обновления включают:

- вычисление $\eta$
- вычисление *unclipped* $\alpha_j$
- ограничение $\alpha_j$ в диапазоне $[L, H]$ (*clip*)
- обновление $\alpha_i$
- обновление смещения $b$

---

### Предсказание (Prediction)

Предсказание для точки $x$ вычисляется как:

$$
f(x) = \sum_{i=1}^{n} \alpha_i y_i K(x_i, x) + b
$$

Во время обучения это значение приходится вычислять часто (используя **кэш ошибок** и **кэш ядра**).  
В худшем случае это даёт вычислительную сложность **$O(n)$ на одну точку**.

Однако после того как большинство $\alpha_i$ становятся нулями, вычисление зависит только от числа **опорных векторов (Support Vectors)**.

---

### KKT-условия

Опорные векторы определяются условиями **KKT (Karush-Kuhn-Tucker)**.

- Если  

$$
\alpha_i = 0
$$

то

$$
y_i f(x_i) \ge 1
$$

- Если  

$$
0 < \alpha_i < C
$$

то

$$
y_i f(x_i) = 1
$$

- Если  

$$
\alpha_i = C
$$

то

$$
y_i f(x_i) \le 1
$$

Точки с $\alpha_i > 0$ являются **support vectors**.

---

### Практические особенности реализации

При реализации SVM важно учитывать:

- **численную устойчивость** (например, случаи $\eta \approx 0$)
- вычисление и усреднение **bias $b$** по внутренним опорным векторам
- использование **kernel cache** для ускорения вычислений
- технику **shrinking** (исключение точек, которые уже удовлетворяют KKT)
- эвристики выбора пары $(i, j)$ для обновления

Все эти методы критически важны для эффективной реализации алгоритма.

Практическая реализация алгоритма доступна в библиотеке **LIBSVM**, основанной на оригинальной идее **SMO**, предложенной **John Platt**.

## Обновление пары $\alpha_i, \alpha_j$ в алгоритме SMO

Алгоритм **SMO (Sequential Minimal Optimization)** обновляет две переменные 
$\alpha_i$ и $\alpha_j$ за одну итерацию.  
Это позволяет сохранить ограничение:

$$
\sum_{i=1}^{n} \alpha_i y_i = 0
$$

---

# 1. Вычисление ошибки

Сначала вычисляются ошибки для выбранных точек:

$$
E_i = f(x_i) - y_i
$$

$$
E_j = f(x_j) - y_j
$$

где

$$
f(x) = \sum_{k=1}^{n} \alpha_k y_k K(x_k, x) + b
$$

---

# 2. Вычисление $\eta$

$$
\eta = K(x_i,x_i) + K(x_j,x_j) - 2K(x_i,x_j)
$$

или кратко

$$
\eta = K_{ii} + K_{jj} - 2K_{ij}
$$

---

# 3. Вычисление нового $\alpha_j$ (без ограничений)

Сначала вычисляется **unclipped значение**:

$$
\alpha_j^{new\_unc} =
\alpha_j +
\frac{y_j (E_i - E_j)}{\eta}
$$

---

# 4. Ограничения $L$ и $H$

Чтобы соблюсти ограничение $0 \le \alpha_j \le C$,  
вычисляются границы $L$ и $H$.

### Если $y_i \neq y_j$

$$
L = \max(0, \alpha_j - \alpha_i)
$$

$$
H = \min(C, C + \alpha_j - \alpha_i)
$$

### Если $y_i = y_j$

$$
L = \max(0, \alpha_i + \alpha_j - C)
$$

$$
H = \min(C, \alpha_i + \alpha_j)
$$

---

# 5. Clip (ограничение $\alpha_j$)

$$
\alpha_j^{new} =
\begin{cases}
H & \text{если } \alpha_j^{new\_unc} > H \\
L & \text{если } \alpha_j^{new\_unc} < L \\
\alpha_j^{new\_unc} & \text{иначе}
\end{cases}
$$

---

# 6. Обновление $\alpha_i$

Чтобы сохранить ограничение

$$
\sum \alpha_i y_i = 0
$$

обновляется $\alpha_i$:

$$
\alpha_i^{new}
=
\alpha_i +
y_i y_j (\alpha_j - \alpha_j^{new})
$$

---

# 7. Обновление bias $b$

Вычисляются два кандидата:

### $b_1$

$$
b_1 =
b - E_i
- y_i(\alpha_i^{new} - \alpha_i)K_{ii}
- y_j(\alpha_j^{new} - \alpha_j)K_{ij}
$$

### $b_2$

$$
b_2 =
b - E_j
- y_i(\alpha_i^{new} - \alpha_i)K_{ij}
- y_j(\alpha_j^{new} - \alpha_j)K_{jj}
$$

---

# 8. Выбор $b$

Выбор значения bias:

- если  

$$
0 < \alpha_i^{new} < C
$$

то

$$
b = b_1
$$

- иначе если  

$$
0 < \alpha_j^{new} < C
$$

то

$$
b = b_2
$$

- иначе

$$
b = \frac{b_1 + b_2}{2}
$$

---

# Итог

После обновления:

- обновляются $\alpha_i$ и $\alpha_j$
- обновляется $b$
- пересчитываются ошибки $E_i$

Процесс повторяется до тех пор, пока **KKT-условия** не будут выполнены для всех точек.